# T4 + T5 — chỉ mục dense, truy hồi 3 nhánh, gộp RRF (1 người)

**Cài đặt notebook:** Accelerator = **GPU T4** (chỉ để mã hoá 1.200 câu hỏi), Internet = **On**.
**Add Input:** `mir-code`, `mir-data`, `mir-bm25`, `mir-emb-m0`, `mir-emb-m1`, `mir-emb-m2`.

**Cách chạy:**
1. Mở notebook, chạy ô (1)–(3). Đọc bảng Recall@100 ở ô (3), chọn `NLIST`/`NPROBE` ở ô (4), ghi bảng vào
   `docs/NOTES.md`. Trên toàn kho, giữ tỷ lệ nprobe/nlist của cặp đã chọn.
2. Đặt `RUN_TUNE = False` → *Save Version* → *Save & Run All (Commit)* để có output.

**Đầu ra:** `runs/` (run từng nhánh, 7 cấu hình A–G, gộp có trọng số) và `index/structure/dense.faiss` → tab *Output*
→ *New Dataset* tên **`mir-runs`**.
Bộ nhớ: IVF-SQ8 cho khoảng 8,5 triệu vector 1024 chiều cần khoảng 9 GB RAM (giới hạn khoảng 30 GB). Nhánh LSR đọc
thẳng từng khối emb, không cần dựng chỉ mục riêng.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây; phần còn lại giống nhau ở mọi notebook.
NB_VERSION = 't5-v2 (05/10/2026)'
RUN_TUNE = True             # ô (3): đo nlist/nprobe. Đặt False khi Save & Run All sau khi đã chọn
OVERRIDES = {}

import glob, os, shutil, sys
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')          # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mir/__init__.py', recursive=True), key=len)
assert hits, 'Chưa thêm dataset mir-code (Add Input → Your Datasets)'
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)                     # luôn chép lại bản mir-code mới nhất
shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mir.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, OVERRIDES, corpus='full')
EMB = ','.join(f'{d}/emb/structure' for d in locate_all('emb/structure'))
BM25 = locate('bm25/stats.json')            # thư mục chứa bm25/ (dataset mir-bm25)
print('emb:', EMB, '\nbm25:', BM25)

In [ ]:
# (2) Mã hoá 1.200 câu hỏi (dense + sparse), dùng chung cho nhánh dense, LSR và ô (3)
sh(f'python -m mir.retrieve qencode --config {K.cfg} --device cuda:0', 'qencode')

In [ ]:
# (3) Chọn nlist/nprobe: Recall@100 của IVF-SQ8 so với IndexFlatIP trên mẫu ~100k đoạn, truy vấn = câu hỏi thật
if RUN_TUNE:
    sh(f'python -m mir.index dense-tune --config {K.cfg} --emb {EMB} --qemb {K.work}/qemb/query.npz', 'dense_tune')

In [ ]:
# (4) Dựng chỉ mục dense IVF-SQ8 cho toàn kho
NLIST, NPROBE = 16384, 64   # ← sửa theo bảng ở ô (3); hai giá trị này là giá trị tạm, chưa đo
K.set({'index.dense.nlist': NLIST, 'index.dense.nprobe': NPROBE})
sh(f'python -m mir.index dense --config {K.cfg} --emb {EMB}', 'dense_build')
print(open(f'{K.work}/index/structure/dense.json').read())

In [ ]:
# (5) Truy hồi 3 nhánh (độ sâu 1000): dense + LSR dùng chỉ mục vừa dựng và emb; BM25 dùng chỉ mục của mir-bm25.
#     Rồi gộp: 7 cấu hình RRF A–G + gộp có trọng số
sh(f'python -m mir.retrieve run --config {K.cfg} --branch dense,lsr --emb {EMB}', 'retrieve_dense_lsr')
sh(f'python -m mir.retrieve run --config {K.cfg} --branch bm25 --index {BM25}', 'retrieve_bm25')
sh(f'python -m mir.fuse --config {K.cfg}', 'fuse')
report('t5')